In [42]:
%load_ext autoreload
%autoreload 2
import pandas as pd
import os

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [43]:
PDF_PATH = "/home/tchakrab/gb-scratch/wvs/data/WVS_TS_codebook.pdf"

import pdfplumber
import re
from typing import Dict, List, Tuple

def extract_pdf_text(pdf_path: str) -> str:
    """Extract raw text from all pages of a PDF."""
    all_text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text(x_tolerance=1, y_tolerance=1)
            if text:
                all_text += text + "\n\n"
    return all_text

In [44]:
variable_lookup_df = pd.read_csv('/home/tchakrab/gb-scratch/wvs/data/WVS_Time_Series_Vars.csv')
var_to_title = variable_lookup_df.set_index('Variable')['Title'].to_dict()
raw_text = extract_pdf_text(PDF_PATH)

In [45]:
PDF_PATH = "/home/tchakrab/gb-scratch/wvs/data/WVS_TS_codebook.pdf"

import pdfplumber
import re
from typing import Dict, List, Tuple


# ============================================================
# 1. PDF extraction
# ============================================================

def extract_pdf_text(pdf_path: str) -> str:
    """Extract raw text from all pages of a PDF."""
    all_text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text(x_tolerance=1, y_tolerance=1)
            if text:
                all_text += text + "\n\n"
    return all_text


# ============================================================
# 2. Cleaning
# ============================================================

def clean_page_artifacts(text: str) -> str:
    """Remove headers, footers, and page numbers WITHOUT destroying newlines."""

    # remove standalone page numbers like:
    # \n - 124 - \n
    text = re.sub(r'\n\s*-\s*\d+\s*-\s*\n', '\n', text)

    # remove inline page numbers like " - 123 -"
    text = re.sub(r'\s*-\s*\d+\s*-\s*', ' ', text)

    # remove report footer/header text (even if inline)
    text = re.sub(
        r'\s*WVS Time Series 1981\s*2022 Variables Report V[\d\.]+\s*',
        ' ',
        text
    )

    # normalize spaces BUT KEEP newlines
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)

    return text.strip()

def strip_report_artifact(text: str) -> str:
    return re.sub(
        r'\s*WVS Time Series 1981\s*2022 Variables Report V[\d\.]+\s*',
        '',
        text
    ).strip()


# ============================================================
# 3. Split into variable blocks
# ============================================================
import re
from typing import List

def split_variable_blocks(text: str) -> List[str]:
    """
    Split cleaned text into variable blocks.

    Robust to cases where a new variable header gets glued mid-line (e.g.,
    "... -5 Missing; Unknown A124_01.- Neighbours ...") by forcing a newline
    before any embedded header first, then doing the usual line-start split.
    """
    # 1) Ensure any variable header that appears mid-line starts on a new line.
    #    This handles pdfplumber "glue" across page/column boundaries.
    header_anywhere = re.compile(r'\s+([A-Z][A-Z0-9_]{1,10}\.-\s+)')
    text = header_anywhere.sub(r'\n\1', text)

    # 2) Now split only on headers at the start of a line (after a newline).
    variable_split_pattern = re.compile(
        r'(\n[A-Z][A-Z0-9_]{1,10}\.-\s+.*)',
        re.MULTILINE
    )

    marked_text = variable_split_pattern.sub(
        r'\n---VAR_START---\1',
        text
    )

    return marked_text.split('---VAR_START---')[1:]



# ============================================================
# 4. Parse a single variable block
# ============================================================

def parse_variable_block(
    block: str,
    target_ids: List[str]
) -> Tuple[str | None, Dict[str, str | List[str]] | None]:
    """Parse one variable block into (var_id, data)."""

    lines = [l.strip() for l in block.split("\n") if l.strip()]
    if not lines:
        return None, None

    # Header: "E220.- <label>"
    m = re.match(r'^([A-Z][A-Z0-9_]{1,10})\.-\s+', lines[0])
    if not m:
        return None, None

    var_id = m.group(1)
    if var_id not in target_ids:
        return None, None

    question_lines = []
    choices = []

    in_choices = False
    pending_code = None

    for line in lines[1:]:

        # Choices marker
        if line == f"({var_id})":
            in_choices = True
            continue

        # ---------------- QUESTION ----------------
        if not in_choices:
            if (
                "WVS Time Series" not in line
                and "Variables Report" not in line
                and var_id not in line
            ):
                question_lines.append(line)
            continue

        # ---------------- CHOICES ----------------
        if pending_code is not None and not re.match(r"^-?\d+\b", line):
            choices.append(f"{pending_code} {line}".strip())
            pending_code = None
            continue

        m_code = re.match(r"^(-?\d+)\b\s*(.*)$", line)
        if m_code:
            code, rest = m_code.groups()
            rest = rest.strip()
            if rest:
                choices.append(f"{code} {rest}")
            else:
                pending_code = code
            continue

        if choices:
            choices[-1] = f"{choices[-1]} {line}".strip()

    if pending_code is not None:
        choices.append(pending_code)

    raw_question = " ".join(question_lines).strip()

    raw_question = strip_report_artifact(raw_question)
    choices = [strip_report_artifact(c) for c in choices]

    return var_id, {
        "question": raw_question,
        "choices": choices
    }


# ============================================================
# 5. Question normalization (anchor logic)
# ============================================================

def clean_label_for_insertion(label: str) -> str:
    """
    Remove category prefixes like 'Democracy:' and trailing punctuation.
    """
    # Remove leading category like "Democracy:"
    label = re.sub(r'^[A-Za-z\s]+:\s*', '', label)

    # Remove trailing colon(s)
    label = re.sub(r':\s*$', '', label)

    return label.strip(":").strip()

def replace_readout_with_label(
    question: str,
    label: str
) -> str:
    pattern = r'\(read out and code one answer for each\)'
    if not re.search(pattern, question, flags=re.IGNORECASE):
        return question

    clean_label = clean_label_for_insertion(label)

    # Ensure exactly one newline before label
    return re.sub(
        pattern,
        f"\n{clean_label}",
        question,
        flags=re.IGNORECASE
    ).strip(":").strip()


def normalize_democracy_question(raw_question: str) -> str:
    """Split instruction vs real question using normalized anchor."""
    norm = re.sub(r"\s+", " ", raw_question).strip()

    ANCHOR = (
        "and 10 means it definitely is “an essential characteristic of democracy"
    )

    idx = norm.find(ANCHOR)
    if idx == -1:
        return raw_question

    instruction = norm[: idx + len(ANCHOR)].strip()
    real_question = norm[idx + len(ANCHOR) :].strip()

    return f"{instruction}\n\n{real_question}"




# ============================================================
# 6. High-level orchestration
# ============================================================

def parse_wvs_questions(
    pdf_path: str,
    raw_text: str,
    target_ids: List[str],
    var_to_title: Dict[str, str]
) -> Dict[str, Dict[str, str | List[str]]]:
    """
    Full pipeline: PDF -> cleaned text -> variable blocks -> parsed questions.
    """

    if pdf_path == None:
        raw_text = raw_text
    else:
        raw_text = extract_pdf_text(pdf_path)
    cleaned_text = clean_page_artifacts(raw_text)
    blocks = split_variable_blocks(cleaned_text)

    results = {}

    for block in blocks:
        var_id, data = parse_variable_block(block, target_ids)
        if var_id is None:
            continue
        
        data["question"] = normalize_democracy_question(data["question"])
        
        if "(read out and code one answer for each)" in data["question"]:
            label = var_to_title.get(var_id)
            if label:
                data["question"] = replace_readout_with_label(
                    data["question"],
                    label
                )

        results[var_id] = data

    return results

In [46]:
parsed_data = parse_wvs_questions(pdf_path=None, raw_text=raw_text, target_ids=list(var_to_title.keys()), var_to_title=var_to_title)
id = 'A124_01'
print(parsed_data[id])

{'question': 'On this list are various groups of people. Could you please mention any that you would not like to have as neighbors?', 'choices': ['0 Not mentioned', '1 Mentioned', "-1 Don't know", '-2 No answer', '-4 Not asked']}


In [47]:
len(parsed_data), len(var_to_title)

(1034, 1048)

In [48]:
system_prompt = '''You are a helpful assistant.'''
for id in parsed_data:
    if id[0] in ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I']:
        print(f"{id}: {parsed_data[id]['question']}")
        print(parsed_data[id]['choices'])
        print(var_to_title[id])

COW_NUM: Country code CoW numeric
[]
CoW country code numeric
COW_ALPHA: CoW country code alpha
[]
CoW country code alpha
A001: For each of the following aspects, indicate how important it is in your life. Would you say it is very important, rather important, not very important or not important at all Family
['1 Very important', '2 Rather important', '3 Not very important', '4 Not at all important', "-1 Don't know", '-2 No answer', '-4 Not asked', '-5 Missing; Not available']
Important in life: Family
A002: For each of the following aspects, indicate how important it is in your life. Would you say it is very important, rather important, not very important or not important at all Friends
['1 Very important', '2 Rather important', '3 Not very important', '4 Not at all important', "-1 Don't know", '-2 No answer', '-4 Not asked', '-5 Missing; Not available']
Important in life: Friends
A003: For each of the following aspects, indicate how important it is in your life. Would you say it is ve

In [49]:
with open('/home/tchakrab/gb-scratch/wvs/code/prompts/system/create_survey.txt', 'r') as f:
    system_prompt = eval(f.read())
with open('/home/tchakrab/gb-scratch/wvs/code/prompts/user/create_survey.txt', 'r') as f:
    user_prompt = f.read()
prompts = []
ids = {}
for id in parsed_data:
    if id[0] in ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I']:
        question = parsed_data[id]['question']
        choices = parsed_data[id]['choices']
        label = var_to_title[id]
        prompts.append(eval(user_prompt))
        ids[prompts[-1]] = id



In [50]:
from thread_gpt_suite.thread_gpt_mp_handler import ThreadGPTMPHandler
handler = ThreadGPTMPHandler(api_key=os.environ.get("OPENAI_API_KEY"), num_worker=350)

In [51]:
batch = [
    {
        "questions": [prompt],          # single-turn
        "model_name": "gpt-4.1-mini",    # change if needed
        "task_desc": system_prompt      # system prompt
    }
    for prompt in prompts
]

# Add and process (same API)
handler.add_batch(batch)
results = handler.process(rerun_on_error=True)

Processing Batch: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 916/916 [00:05<00:00, 181.35conv/s]


In [52]:
import json

data = []
for item in results:
    for prompt, response in item.items():
        id = ids[prompt]
        label = var_to_title[id]
        choices = parsed_data[id]['choices']

        data.append({
            id: {
                "label": label,
                "choices": choices,
                "parsed_question": response,
                "original_question": parsed_data[id]['question']
            }
        })

# Identify and combine questions with "choose up to" (case insensitive)
special_ids = []
for item in data:
    id = list(item.keys())[0]
    if 'choose up to' in item[id]['original_question'].lower():
        special_ids.append(id)

if special_ids:
    # Sort IDs for combined key
    sorted_special_ids = sorted(special_ids)
    combined_id = '_'.join(sorted_special_ids)
    
    # Extract qualities from labels (after ': ')
    qualities = []
    for sid in sorted_special_ids:
        for item in data:
            if list(item.keys())[0] == sid:
                quality = item[sid]['label'].split(': ')[1]
                qualities.append(quality)
                break
    
    # Get base parsed and original questions from the first special item
    base_parsed = None
    base_original = None
    for item in data:
        if list(item.keys())[0] == sorted_special_ids[0]:
            base_parsed = item[sorted_special_ids[0]]['parsed_question'].split('\nQuality: ')[0]
            # Assume original ends with the specific quality; adjust if needed
            quality_in_original = item[sorted_special_ids[0]]['original_question'].split(' Please choose up to five. ')[1]
            base_original = item[sorted_special_ids[0]]['original_question'].replace(' ' + quality_in_original, '')
            break
    
    # Build combined parsed question
    combined_parsed = base_parsed + '\nQualities: ' + ', '.join([f'{chr(65+i)}: {q}' for i, q in enumerate(qualities)])
    
    # Create combined item
    combined_item = {
        combined_id: {
            "label": 'Important child qualities',
            "choices": "Pick up to 5 letters corresponding to the qualities mentioned in the question",
            "parsed_question": combined_parsed,
            "original_question": base_original
        }
    }
    
    # Add combined item and remove individual special items
    data.append(combined_item)
    data = [item for item in data if list(item.keys())[0] not in special_ids]

with open('/home/tchakrab/gb-scratch/wvs/data/parsed_survey_data.jsonl', 'w') as f:
    for row in data:
        f.write(json.dumps(row) + "\n")